<a href="https://colab.research.google.com/github/monowar-mukul/AI-102_Azure-AI-Engineer-Associate--Labs/blob/main/Fitness_Nutrition_MultiAgent_LangGraph_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Multi-Agent Fitness & Nutrition Coach — Groq + LangGraph Lab**

Purpose of this lab: practice a multi-agent LangGraph pattern (router/orchestration, shared state, human-in-the-loop, checkpoint memory) on a real scenario so you can confirm you understand the pattern rather than just having copied it.

**Scenario**

A fitness app wants an assistant that helps users with two things:

* **Workout Advisor** — suggests exercise routines / workout plans.
* **Nutrition Advisor** — suggests meals / diet plans that support the workout goal.

A human-in-the-loop node pauses after every agent turn so a real person can:

* ask a follow-up question,
* redirect to the other advisor (e.g. "now suggest food"),
* or add a constraint (e.g. "I'm vegetarian", "I have a knee injury") that the next agent must respect.

**Architecture**

* A router (`route_after_human`) decides whether to go to the Workout Advisor or Nutrition Advisor — this is the hierarchical / orchestration pattern.
* State (`last_active_agent`, full conversation history) is tracked via LangGraph's `StateGraph` + `MemorySaver` checkpointing, so the graph "remembers" the thread across turns.
* A human node pauses the graph using LangGraph's `interrupt()` / `Command(resume=...)` pattern so a person can approve, clarify, or redirect before the workflow resumes.
* Two tools: `get_workout_recommendation` and `get_meal_plan_recommendation` — both use simple rule-based logic here, but could be handed fully to the LLM's own knowledge instead.

---


## Step 1 — Install dependencies

Install the following packages: langchain, langgraph, langchain-groq. If you're on Colab, go to Runtime → Change runtime type → T4 GPU first (optional, mainly helps if you later swap in a locally-hosted model; Groq itself runs on Groq's own cloud hardware, not your Colab GPU).


In [1]:
# Install/upgrade the libraries this lab needs.
# langchain        -> agent framework (create_agent, tools, messages)
# langgraph        -> the graph/state-machine engine that wires our agents + human node together
# langchain-groq   -> lets LangChain talk to Groq-hosted open-source models
!pip install -U langchain langgraph langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17


## Step 2 — Load your Groq API key

You'll be prompted to paste your key. It is only stored in this Colab session's environment variables,
never written to disk or shared.


In [2]:
import os
from getpass import getpass

# Only ask for the key if it isn't already set in the environment.
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("GROQ_API_KEY loaded:", bool(os.getenv("GROQ_API_KEY")))


Enter your Groq API key: ··········
GROQ_API_KEY loaded: True


## Step 3 — Imports

- `ChatGroq` — the chat model wrapper that calls Groq's API.
- `create_agent` — LangChain v1's current API for building a tool-using agent (replaces the older,
  deprecated `create_react_agent` from `langgraph.prebuilt`).
- `tool` — decorator that turns a plain Python function into something an LLM agent can call.
- `MessagesState`, `StateGraph`, `START` — the core LangGraph building blocks for defining our graph and
  its shared state.
- `Command`, `interrupt` — how a graph node can (a) update state + decide the next node (`Command`), and
  (b) pause the whole graph to wait for human input (`interrupt`).
- `MemorySaver` — a checkpointer that persists graph state per "thread", so the conversation survives
  across multiple `graph.stream(...)` calls.
- `HumanMessage`, `AIMessage` — LangChain's typed chat message objects.


In [3]:
from typing import Literal

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool

from langgraph.graph import MessagesState, StateGraph, START
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver

from langchain_core.messages import HumanMessage, AIMessage

print("Imports successful")


Imports successful


### Step 4 — Initialize the Groq model

We use the `openai/gpt-oss-120b` open-source model on Groq, since it supports tool calling / agentic workflows. `temperature=0` keeps answers deterministic and easier to grade in class.

In [4]:
# Create the chat model that both of our agents will share.
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=2,
)

print("Model initialized:", model.model_name)


Model initialized: openai/gpt-oss-120b


In [5]:
# Quick sanity check that the model + API key actually work, before we build anything complicated.
response = model.invoke("Say hello in one short sentence.")
print(response.content)


Hello!


### Step 5 — Define the shared multi-agent state

`MessagesState` already gives us a `messages` list (the conversation history) for free. We extend it with one extra field, `last_active_agent`, so the human node and router know which advisor was speaking most recently (used as a fallback when the user's next message doesn't clearly belong to either advisor).

In [6]:
class MultiAgentState(MessagesState):
    # Tracks which advisor last responded, so the router has a sensible default
    # when the user's next message is ambiguous (e.g. "tell me more").
    last_active_agent: Literal["workout_advisor", "nutrition_advisor"]


### Step 6 — Define the two tools

These are plain Python functions decorated with `@tool`, which makes them callable by the LLM agents. Each tool uses simple keyword-based rules here (easy to demo and debug in class) — in a production system you could instead let the LLM answer purely from its own knowledge, or call a real fitness/nutrition API.

In [7]:
@tool
def get_workout_recommendation(query: str) -> str:
    """Return a workout/exercise plan recommendation based on the user's fitness goal."""
    q = query.lower()

    if "muscle" in q or "strength" in q or "bulk" in q:
        return (
            "For building muscle: a 4-day split — Day 1 Chest/Triceps, Day 2 Back/Biceps, "
            "Day 3 Legs, Day 4 Shoulders/Core. Aim for 3-4 sets of 8-12 reps per exercise, "
            "progressively increasing weight."
        )
    if "weight loss" in q or "fat" in q or "cardio" in q or "lose" in q:
        return (
            "For fat loss: combine 3 days of full-body strength training with 2-3 days of "
            "cardio (running, cycling, or HIIT intervals of 20-30 minutes)."
        )
    if "beginner" in q or "start" in q:
        return (
            "For beginners: start with 3 full-body workouts per week (squats, push-ups, rows, "
            "planks), focusing on form before adding weight."
        )
    if "knee" in q or "injury" in q:
        return (
            "For a knee injury: favor low-impact options like swimming, seated leg extensions "
            "(light weight), and upper-body strength work. Please consult a physio before "
            "resuming squats or running."
        )
    return "I recommend a balanced routine of strength training 3x/week plus 2 cardio sessions."


@tool
def get_meal_plan_recommendation(query: str) -> str:
    """Return a meal/nutrition recommendation based on the user's goal or dietary preference."""
    q = query.lower()

    if "vegetarian" in q or "vegan" in q:
        return (
            "Vegetarian option: lentils, chickpeas, tofu, paneer, Greek yogurt, and quinoa for "
            "protein; add nuts and seeds for healthy fats. Aim for 1.6-2.2g protein per kg "
            "bodyweight if the goal is muscle gain."
        )
    if "muscle" in q or "protein" in q or "bulk" in q:
        return (
            "For muscle-building nutrition: prioritize lean protein (chicken, fish, eggs, "
            "Greek yogurt) at every meal, complex carbs (rice, oats, sweet potato), and a "
            "slight calorie surplus."
        )
    if "fat" in q or "lose" in q or "cut" in q:
        return (
            "For fat loss: eat in a modest calorie deficit, prioritize protein and vegetables "
            "to stay full, and limit added sugars and fried foods."
        )
    if "snack" in q:
        return (
            "Good snack ideas: Greek yogurt with berries, a handful of almonds, hummus with "
            "veggie sticks, or a protein shake."
        )
    return "Aim for balanced meals: a protein source, a complex carb, vegetables, and healthy fats."


### Step 7 — Create the two agents

Each agent is built with `create_agent`, given the model, its one relevant tool, and a role-specific system prompt. Notice each prompt explicitly tells the agent not to answer outside its lane — that's what makes the router (not the LLM itself) responsible for handoffs.


In [8]:
workout_advisor = create_agent(
    model=model,
    tools=[get_workout_recommendation],
    system_prompt=(
        "You are the Workout Advisor. "
        "Help users design exercise routines and workout plans. "
        "Use get_workout_recommendation when a workout suggestion is needed. "
        "Do not provide meal or nutrition advice yourself; the outer LangGraph will route "
        "nutrition requests to the Nutrition Advisor. "
        "Keep answers concise and practical."
    ),
    name="workout_advisor",
)

nutrition_advisor = create_agent(
    model=model,
    tools=[get_meal_plan_recommendation],
    system_prompt=(
        "You are the Nutrition Advisor. "
        "Recommend meals and eating habits that support the user's fitness goal, and respect "
        "any dietary constraints (e.g. vegetarian, vegan, allergies) mentioned earlier in the "
        "conversation. "
        "Use get_meal_plan_recommendation when a meal suggestion is needed. "
        "Do not design workout routines yourself; that is the Workout Advisor's job. "
        "Keep answers concise and practical."
    ),
    name="nutrition_advisor",
)

print("Workout and Nutrition agents created")


Workout and Nutrition agents created


## Step 8 — Graph node functions for each agent

Each node function takes the current graph `state`, invokes the matching agent on the conversation so
far, and returns a `Command`:
- `update` — how to change the shared state (append the agent's new messages, record which agent just ran).
- `goto` — which node to visit next. Both advisors always hand off to `"human"` next, so a person can react
  before the conversation continues.


In [9]:
def call_workout_advisor(state: MultiAgentState) -> Command:
    # create_agent expects {"messages": [...]}; we pass the full history so it has context.
    response = workout_advisor.invoke({"messages": state["messages"]})

    return Command(
        update={
            "messages": response["messages"],
            "last_active_agent": "workout_advisor",
        },
        goto="human",  # always pause for human input/approval after the agent responds
    )


def call_nutrition_advisor(state: MultiAgentState) -> Command:
    response = nutrition_advisor.invoke({"messages": state["messages"]})

    return Command(
        update={
            "messages": response["messages"],
            "last_active_agent": "nutrition_advisor",
        },
        goto="human",
    )


## Step 9 — Router + human-in-the-loop node

- `route_after_human` is a simple deterministic keyword router (easy to reason about for teaching
  purposes — in a production system you might use an LLM classifier instead).
- `human_node` calls `interrupt(...)`, which **pauses the graph** and waits for external input. When you
  resume the graph with `Command(resume=<text>)`, that text becomes `user_input` inside this function.


In [10]:
def route_after_human(user_input: str, active_agent: str) -> str:
    """Simple deterministic router for this teaching example."""
    text = user_input.lower()

    nutrition_words = [
        "eat", "meal", "food", "diet", "nutrition", "snack",
        "vegetarian", "vegan", "protein", "calories",
    ]
    workout_words = [
        "workout", "exercise", "routine", "gym", "training",
        "cardio", "strength", "muscle", "reps", "sets",
    ]

    if any(word in text for word in nutrition_words):
        return "nutrition_advisor"

    if any(word in text for word in workout_words):
        return "workout_advisor"

    # Ambiguous input (e.g. "tell me more", "I'm vegetarian, adjust that") -> stick with
    # whichever advisor was active last, so context/constraints carry over correctly.
    return active_agent


def human_node(state: MultiAgentState) -> Command:
    # interrupt() pauses graph execution here until the caller resumes it with Command(resume=...).
    user_input = interrupt(value="Ready for user input.")

    active_agent = state.get("last_active_agent", "workout_advisor")
    next_agent = route_after_human(user_input, active_agent)

    return Command(
        update={
            "messages": [HumanMessage(content=user_input)],
        },
        goto=next_agent,
    )


## Step 10 — Build and compile the state graph

We register the three nodes (`workout_advisor`, `nutrition_advisor`, `human`), point `START` at the
Workout Advisor (every conversation begins there), and compile with a `MemorySaver` checkpointer so the
conversation state persists across multiple `graph.stream(...)` calls that share the same `thread_id`.


In [11]:
builder = StateGraph(MultiAgentState)

builder.add_node("workout_advisor", call_workout_advisor)
builder.add_node("nutrition_advisor", call_nutrition_advisor)
builder.add_node("human", human_node)

# Every new conversation starts with the Workout Advisor.
builder.add_edge(START, "workout_advisor")

# MemorySaver keeps the conversation state in memory, keyed by thread_id, across multiple
# graph.stream() calls -- this is what lets Command(resume=...) continue the same conversation.
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled successfully")


Graph compiled successfully


## Step 11 — Run the 3-turn demo conversation

This mirrors the Travel Advisor lab's test pattern:

1. **Turn 1** (workout goal) → should route to `workout_advisor` (the graph's default start node).
2. **Turn 2** (nutrition question) → the human node routes to `nutrition_advisor` based on keywords.
3. **Turn 3** (adds a vegetarian constraint + asks for a snack) → stays with `nutrition_advisor`, and
   should reflect the vegetarian constraint using conversation context.

Each `Command(resume=...)` continues the *same* thread (`thread_config`), so state and history persist.


In [12]:
import uuid

thread_config = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

inputs = [
    {
        "messages": [
            {
                "role": "user",
                "content": "I want to build muscle and lose some fat, suggest a workout plan"
            }
        ]
    },
    Command(
        resume="What should I eat to support this workout plan?"
    ),
    Command(
        resume="I'm vegetarian, can you adjust the meal suggestions and also give me a quick snack idea?"
    ),
]

for idx, user_input in enumerate(inputs):
    print(f"\n--- Conversation Turn {idx + 1} ---\n")

    if isinstance(user_input, dict):
        print("User:", user_input["messages"][0]["content"])
    else:
        print("User:", user_input.resume)

    # stream_mode="updates" yields incremental state changes as each node runs,
    # which lets us print each agent's reply as it happens.
    for update in graph.stream(
        user_input,
        config=thread_config,
        stream_mode="updates",
    ):
        for node_id, value in update.items():
            if not isinstance(value, dict):
                continue

            messages = value.get("messages", [])
            if not messages:
                continue

            last_message = messages[-1]

            if isinstance(last_message, AIMessage):
                print(f"{node_id}: {last_message.content}")



--- Conversation Turn 1 ---

User: I want to build muscle and lose some fat, suggest a workout plan
workout_advisor: **4‑Day Upper/Lower Split (8‑12 rep range, 3‑4 sets each)**  

| Day | Main Focus | Sample Exercises |
|-----|------------|-------------------|
| **1** | **Upper – Push** (Chest / Shoulders / Triceps) | Bench Press, Incline DB Press, Overhead Press, Lateral Raises, Dips or Triceps Push‑downs |
| **2** | **Lower** (Quads / Hamstrings / Glutes / Core) | Squats or Front Squats, Romanian Deadlifts, Leg Press, Walking Lunges, Plank / Hanging Leg‑Raises |
| **3** | **Upper – Pull** (Back / Biceps) | Pull‑ups/Lat Pulldowns, Barbell Row, Seated Cable Row, Face Pulls, DB Curls, Hammer Curls |
| **4** | **Full‑Body / Conditioning** | Deadlift or Power Clean, Bulgarian Split Squat, Push‑Press, Farmer’s Walk, 15‑20 min HIIT (e.g., 30 s sprint/90 s jog) |

**Guidelines**

1. **Progressive overload** – add 2.5–5 lb (1–2 kg) each week if you can hit the top rep range with good form.  

## What this lab tests (vs. the Travel Advisor lab)

| Feature | Travel Advisor lab | Fitness/Nutrition lab |
|---|---|---|
| Router pattern | keyword router: hotel vs. travel | keyword router: nutrition vs. workout |
| Shared state | `last_active_agent`, messages | same shape, new domain |
| Human-in-the-loop | `interrupt()` / `Command(resume=...)` | identical mechanism |
| Tools | 2 rule-based tools | 2 rule-based tools |
| Cross-turn context | "the first one" (hotel) | "I'm vegetarian" constraint carried into later turn |
| Checkpointing | `MemorySaver`, single thread | same |

If your 3-turn run above correctly (a) starts at the Workout Advisor, (b) hands off to the Nutrition
Advisor on the food question, and (c) applies the vegetarian constraint on turn 3 — you've confirmed you
understand the router + state + human-in-the-loop pattern well enough to reuse it on a new domain, not
just recognize it in the original example.

**Try it yourself:** swap `resume=...` for `resume=input("Your query: ")` on any turn to type your own
follow-up interactively, the same way the original notebook allows.
